# TorlakTag: Lemma-Only Model (ByT5 seq2seq)

Trains a **generative** lemmatizer using `google/byt5-large` (or `byt5-base` for speed).  
Seq2seq generation handles OOV dialect forms that a fixed-vocabulary classifier cannot.  
After training the best checkpoint is used to **replace the LEMMA column** in every file under `final_conllu/`.

**Input format:** `lemmatize: {left_5_words} [X] {form} [/X] {right_5_words}`  
**Output:** `{lemma}`

**Improvements over the original `finetune_lemma.ipynb`:**
- Larger model (`byt5-large`, configurable)
- Wider context window (5 words each side)
- FP16 mixed-precision + gradient accumulation
- Cosine LR schedule with warmup
- Consistent token normalization (matches `final_conllu` forms)
- Full pipeline to re-lemmatize all `final_conllu/` files

In [ ]:
!pip install -q transformers torch sentencepiece python-Levenshtein

from google.colab import drive
drive.mount('/content/drive')

import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os, re, json, random, time
from pathlib import Path
import numpy as np
import torch

# ── PATHS ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT     = Path('/content/drive/MyDrive/TorlakTag')
TRAIN_TSV      = DRIVE_ROOT / 'dataset' / 'tor_train.tsv'
DEV_TSV        = DRIVE_ROOT / 'dataset' / 'tor_dev.tsv'
TEST_TSV       = DRIVE_ROOT / 'dataset' / 'tor_test.tsv'

FINAL_CONLLU_DIR  = DRIVE_ROOT / 'final_conllu'
OUTPUT_CONLLU_DIR = DRIVE_ROOT / 'final_conllu_relemmad'
OUTPUT_CONLLU_DIR.mkdir(parents=True, exist_ok=True)

# Auto-resume: find the most recent byt5_lemma_* directory that has a checkpoint.
# Only create a new timestamped directory if nothing resumable exists.
_models_root = DRIVE_ROOT / 'models'
_models_root.mkdir(parents=True, exist_ok=True)
_existing = sorted(
    [d for d in _models_root.glob('byt5_lemma_*') if (d / 'last_ckpt.pt').exists()],
    key=lambda d: d.name
)
if _existing:
    SAVE_DIR = _existing[-1]
    print(f'📂 Found existing checkpoint → will resume from: {SAVE_DIR}')
else:
    SAVE_DIR = _models_root / f'byt5_lemma_{time.strftime("%Y%m%d_%H%M%S")}'
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f'📂 No checkpoint found → new run: {SAVE_DIR}')

# ── MODEL ─────────────────────────────────────────────────────────────────────
MODEL_NAME = 'google/byt5-large'

# ── TRAINING HYPERPARAMS ──────────────────────────────────────────────────────
CONTEXT_WINDOW = 5
MAX_SRC_LEN    = 192
MAX_TGT_LEN    = 64
BATCH_SIZE     = 32
GRAD_ACCUM     = 2        # effective batch = 64
LR             = 5e-5
WEIGHT_DECAY   = 0.01
EPOCHS         = 15
WARMUP_RATIO   = 0.06
GRAD_CLIP      = 1.0
PATIENCE       = 4

# bfloat16: same dynamic range as float32 (no overflow → no NaN), A100-native speed.
AMP_DTYPE      = torch.bfloat16   # set to None to use full FP32

# ── INFERENCE ─────────────────────────────────────────────────────────────────
INFER_BATCH    = 256
NUM_BEAMS      = 4

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print(f'GPU: {torch.cuda.get_device_name(0)}')

## Token normalisation
Identical to the multirun notebook so training forms match the forms already in `final_conllu/`.

In [ ]:
def apply_spec_mapping(s: str) -> str:
    if s is None:
        return ''
    s = s.replace('#', '')
    s = s.replace('W', 'Ə').replace('w', 'ə')
    s = s.replace('1', 'ḱ').replace('6', 'ḱ')
    s = s.replace('2', 'ǵ')
    s = s.replace('3', 'č')
    s = s.replace('x', 'š').replace('X', 'š')
    s = s.replace('5', 'ƨ')
    s = s.replace('ššš', 'XXX')
    return s

RE_LONGVOWEL = re.compile(r'([aeiouə])\1+')
RE_SPACES    = re.compile(r'\s+')
_STRIP_EDGE  = " \t\r\n\"'\u201c\u201d\u201e`\u00b4.,;:!?(){}[]<>•"

def strip_attached_specials(tok: str) -> str:
    t = tok.strip()
    t = re.sub(r'/+$', '', t)
    t = t.strip(_STRIP_EDGE)
    return t

def normalize_word(tok: str) -> str:
    t = apply_spec_mapping(tok).lower()
    t = RE_LONGVOWEL.sub(r'\1', t)
    t = RE_SPACES.sub(' ', t).strip()
    return t

def norm_form(tok: str) -> str:
    """Normalize a training-data form to match final_conllu forms."""
    w = strip_attached_specials(tok)
    return normalize_word(w) if w else tok.lower().strip()

# Quick sanity check
assert norm_form('pričAla') == 'pričala'
assert norm_form('lazarIčke') == 'lazarIčke'.lower()  # lowercased
print('✅ normalisation OK')

## Data loading

In [ ]:
def load_tsv(path):
    """Parse form\tlemma\txpos TSV; blank / tab-only lines = sentence boundaries."""
    sentences, current = [], []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                if current:
                    sentences.append(current)
                    current = []
            else:
                parts = line.split('\t')
                if len(parts) == 3 and parts[0].strip():
                    current.append({
                        'form':  norm_form(parts[0]),
                        'lemma': normalize_word(strip_attached_specials(parts[1])) or norm_form(parts[0]),
                        'xpos':  parts[2].strip(),
                    })
    if current:
        sentences.append(current)
    return sentences

train_sents = load_tsv(TRAIN_TSV)
dev_sents   = load_tsv(DEV_TSV)
test_sents  = load_tsv(TEST_TSV)

for split, sents in [('train', train_sents), ('dev', dev_sents), ('test', test_sents)]:
    toks = sum(len(s) for s in sents)
    print(f'{split:5s}: {len(sents):5d} sentences  {toks:6d} tokens')

In [ ]:
def make_pairs(sentences, ctx=CONTEXT_WINDOW):
    """
    Build (source, target) pairs for every token.
    source = 'lemmatize: {left} [X] {form} [/X] {right}'
    target = lemma
    Skips tokens with empty forms (punctuation-only rows).
    """
    pairs = []
    for sent in sentences:
        for i, tok in enumerate(sent):
            if not tok['form']:
                continue
            left  = ' '.join(t['form'] for t in sent[max(0, i - ctx): i])
            right = ' '.join(t['form'] for t in sent[i + 1: i + 1 + ctx])
            src   = f"lemmatize: {left} [X] {tok['form']} [/X] {right}".strip()
            pairs.append({'source': src, 'target': tok['lemma']})
    return pairs

train_pairs = make_pairs(train_sents)
dev_pairs   = make_pairs(dev_sents)
test_pairs  = make_pairs(test_sents)

print(f'Pairs — train: {len(train_pairs)}  dev: {len(dev_pairs)}  test: {len(test_pairs)}')
print('\nSample pairs:')
for p in train_pairs[2:5]:
    print(f'  SRC: {p["source"]}')
    print(f'  TGT: {p["target"]}\n')

## Tokenization & Datasets

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('✅ tokenizer loaded:', MODEL_NAME)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class LemmaDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        return self.pairs[i]['source'], self.pairs[i]['target']

def collate_fn(batch):
    """Tokenize and dynamically pad one batch — much faster than pre-encoding everything."""
    sources = [b[0] for b in batch]
    targets = [b[1] for b in batch]

    enc = tokenizer(
        sources,
        max_length=MAX_SRC_LEN, truncation=True,
        padding=True,           # pad to longest in this batch, not global max
        return_tensors='pt',
    )
    lbl_enc = tokenizer(
        targets,
        max_length=MAX_TGT_LEN, truncation=True,
        padding=True,
        return_tensors='pt',
    )
    labels = lbl_enc['input_ids'].clone()
    labels[labels == tokenizer.pad_token_id] = -100   # mask padding in loss
    enc['labels'] = labels
    return enc

train_loader = DataLoader(LemmaDataset(train_pairs), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True, collate_fn=collate_fn)
dev_loader   = DataLoader(LemmaDataset(dev_pairs),   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True, collate_fn=collate_fn)
test_loader  = DataLoader(LemmaDataset(test_pairs),  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True, collate_fn=collate_fn)

print(f'Batches — train: {len(train_loader)}  dev: {len(dev_loader)}  test: {len(test_loader)}')

## Model & Training

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
import math
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from tqdm.auto import tqdm

PAD = tokenizer.pad_token_id

@torch.no_grad()
def evaluate_loader(loader, max_samples=None):
    """Exact-match accuracy via beam decoding."""
    model.eval()
    correct = total = 0
    for batch in loader:
        if max_samples and total >= max_samples:
            break
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].clone()
        lbls[lbls == -100] = PAD

        if AMP_DTYPE is not None:
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                gen = model.generate(ids, attention_mask=mask,
                                     max_new_tokens=MAX_TGT_LEN, num_beams=NUM_BEAMS)
        else:
            gen = model.generate(ids, attention_mask=mask,
                                 max_new_tokens=MAX_TGT_LEN, num_beams=NUM_BEAMS)

        for g, l in zip(gen, lbls):
            pred = tokenizer.decode(g,           skip_special_tokens=True).strip()
            gold = tokenizer.decode(l[l != PAD], skip_special_tokens=True).strip()
            correct += int(pred == gold)
            total   += 1
    return correct / total if total else 0.0

# ── Optimizer & schedule ──────────────────────────────────────────────────────
total_steps  = math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

# bfloat16 doesn't overflow so no GradScaler needed
use_amp = AMP_DTYPE is not None and device.type == 'cuda'

# ── Resume from checkpoint if one exists ─────────────────────────────────────
CKPT_PATH   = SAVE_DIR / 'last_ckpt.pt'
start_epoch = 1
best_dev    = 0.0
bad         = 0
log         = []

if CKPT_PATH.exists():
    print(f'🔄 Resuming from checkpoint: {CKPT_PATH}')
    ckpt        = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch = ckpt['epoch'] + 1
    best_dev    = ckpt['best_dev']
    bad         = ckpt['bad']
    log         = ckpt.get('log', [])
    print(f'   Resuming epoch {start_epoch}  best_dev={best_dev:.4f}  patience={bad}/{PATIENCE}')
else:
    print('🚀 Starting fresh training')

# ── Training loop ─────────────────────────────────────────────────────────────
for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    running = 0.0
    t0      = time.time()
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}'), start=1):
        ids  = batch['input_ids'].to(device, non_blocking=True)
        mask = batch['attention_mask'].to(device, non_blocking=True)
        lbls = batch['labels'].to(device, non_blocking=True)

        if use_amp:
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                loss = model(input_ids=ids, attention_mask=mask, labels=lbls).loss / GRAD_ACCUM
        else:
            loss = model(input_ids=ids, attention_mask=mask, labels=lbls).loss / GRAD_ACCUM

        loss.backward()
        running += float(loss.item()) * GRAD_ACCUM

        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

    dev_acc  = evaluate_loader(dev_loader, max_samples=1000)
    dt       = time.time() - t0
    avg_loss = running / len(train_loader)

    row = {'epoch': epoch, 'train_loss': avg_loss, 'dev_acc_1k': dev_acc, 'minutes': dt/60}
    log.append(row)
    print(f'[{epoch:02d}] loss={avg_loss:.4f}  dev_acc(1k)={dev_acc:.4f}  ({dt/60:.1f}m)')

    if dev_acc > best_dev + 1e-5:
        best_dev = dev_acc
        bad      = 0
        model.save_pretrained(str(SAVE_DIR / 'best'))
        tokenizer.save_pretrained(str(SAVE_DIR / 'best'))
        print(f'  💾 saved best (dev_acc={best_dev:.4f})')
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f'⏹️  Early stop. Best dev_acc={best_dev:.4f}')
            torch.save({'epoch': epoch, 'model': model.state_dict(),
                        'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                        'best_dev': best_dev, 'bad': bad, 'log': log}, CKPT_PATH)
            break

    torch.save({'epoch': epoch, 'model': model.state_dict(),
                'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                'best_dev': best_dev, 'bad': bad, 'log': log}, CKPT_PATH)
    json.dump(log, open(SAVE_DIR / 'train_log.json', 'w'), indent=2)

print('\n✅ Training complete. Best dev_acc:', best_dev)

## Full Test-set Evaluation

In [ ]:
import Levenshtein

# Reload best checkpoint
model = AutoModelForSeq2SeqLM.from_pretrained(str(SAVE_DIR / 'best')).to(device)
tokenizer_best = AutoTokenizer.from_pretrained(str(SAVE_DIR / 'best'))

test_acc = evaluate_loader(test_loader)
print(f'Test exact-match accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')

# Error analysis
model.eval()
errors = []
with torch.no_grad():
    for batch in test_loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].clone()
        lbls[lbls == -100] = PAD
        gen  = model.generate(ids, attention_mask=mask,
                               max_new_tokens=MAX_TGT_LEN, num_beams=NUM_BEAMS)
        for g, l, s in zip(gen, lbls, batch['input_ids']):
            pred = tokenizer.decode(g,           skip_special_tokens=True).strip()
            gold = tokenizer.decode(l[l != PAD], skip_special_tokens=True).strip()
            src  = tokenizer.decode(s,           skip_special_tokens=True).strip()
            if pred != gold:
                errors.append({'src': src, 'pred': pred, 'gold': gold,
                                'edit': Levenshtein.distance(pred, gold)})

avg_edit = sum(e['edit'] for e in errors) / len(errors) if errors else 0.0
print(f'Errors: {len(errors)} / {len(test_pairs)}  avg edit-distance: {avg_edit:.2f}')
print(f'\nTop-20 worst errors:')
print(f'{"source":<50} {"pred":<20} {"gold":<20} edit')
print('-' * 100)
for e in sorted(errors, key=lambda x: -x['edit'])[:20]:
    print(f"{e['src']:<50} {e['pred']:<20} {e['gold']:<20} {e['edit']}")

## Replace LEMMA in all `final_conllu/` files

Reads every `.conllu` from `FINAL_CONLLU_DIR`, replaces only the LEMMA column (column 3, 1-indexed),  
and writes updated files to `OUTPUT_CONLLU_DIR`.  
Comment lines, blank lines, and X-tagged special tokens are left unchanged.

In [ ]:
from typing import List, Tuple

def build_source_strings(forms: List[str], ctx: int = CONTEXT_WINDOW) -> List[str]:
    """Given an ordered list of forms for one sentence, return source strings for each token."""
    srcs = []
    for i, form in enumerate(forms):
        left  = ' '.join(forms[max(0, i - ctx): i])
        right = ' '.join(forms[i + 1: i + 1 + ctx])
        srcs.append(f'lemmatize: {left} [X] {form} [/X] {right}'.strip())
    return srcs

@torch.no_grad()
def predict_lemmas_batch(sources: List[str]) -> List[str]:
    """Run seq2seq inference on a flat list of source strings; return predicted lemmas."""
    lemmas = []
    for i in range(0, len(sources), INFER_BATCH):
        chunk = sources[i: i + INFER_BATCH]
        enc = tokenizer(
            chunk,
            max_length=MAX_SRC_LEN, truncation=True, padding=True,
            return_tensors='pt',
        ).to(device)
        gen = model.generate(
            **enc,
            max_new_tokens=MAX_TGT_LEN,
            num_beams=NUM_BEAMS,
        )
        for g in gen:
            lemmas.append(tokenizer.decode(g, skip_special_tokens=True).strip())
    return lemmas

def parse_token_line(line: str) -> Tuple[bool, List[str]]:
    """
    Returns (is_token_line, fields).
    A token line has a numeric or range ID in field 0 (e.g. '1', '1-2').
    Comment and blank lines are not token lines.
    """
    if not line or line.startswith('#'):
        return False, []
    parts = line.split('\t')
    if len(parts) < 10:
        return False, []
    tok_id = parts[0].strip()
    if not re.match(r'^\d+(-\d+)?$', tok_id):
        return False, []
    return True, parts

def is_special_token(fields: List[str]) -> bool:
    """Special/unintelligible tokens have UPOS == 'X'."""
    return len(fields) >= 4 and fields[3].strip() == 'X'

print('✅ inference helpers ready')

In [ ]:
model.eval()

conllu_files = sorted(FINAL_CONLLU_DIR.glob('*.conllu'))
print(f'Found {len(conllu_files)} conllu files in {FINAL_CONLLU_DIR}')

total_replaced = 0
total_kept     = 0

for fpath in conllu_files:
    lines = fpath.read_text(encoding='utf-8').splitlines()

    # ── Pass 1: collect sentences (groups of token lines between blanks) ──────
    # Each sentence is a list of (line_index, fields) for real token lines.
    sentences_idx = []   # list of list[(line_idx, fields)]
    current       = []
    for li, line in enumerate(lines):
        is_tok, fields = parse_token_line(line)
        if is_tok:
            current.append((li, fields))
        else:
            if current:
                sentences_idx.append(current)
                current = []
    if current:
        sentences_idx.append(current)

    # ── Pass 2: build source strings for non-special tokens ──────────────────
    # Map line_index → predicted lemma (None if special / multiword)
    line2lemma = {}

    for sent_tokens in sentences_idx:
        # Non-special single-token rows (skip multiword spans like 1-2)
        normal = [
            (li, fields)
            for li, fields in sent_tokens
            if '-' not in fields[0] and not is_special_token(fields)
        ]
        if not normal:
            continue

        forms   = [fields[1].strip() for _, fields in normal]
        sources = build_source_strings(forms)
        lemmas  = predict_lemmas_batch(sources)

        for (li, _), pred_lemma in zip(normal, lemmas):
            # Fallback: if model outputs empty string, keep the form
            line2lemma[li] = pred_lemma if pred_lemma else forms[normal.index((li, _))]

    # ── Pass 3: rebuild file with replaced LEMMA column ──────────────────────
    out_lines = []
    for li, line in enumerate(lines):
        if li in line2lemma:
            parts        = line.split('\t')
            parts[2]     = line2lemma[li]          # LEMMA is column index 2
            out_lines.append('\t'.join(parts))
            total_replaced += 1
        else:
            out_lines.append(line)
            is_tok, fields = parse_token_line(line)
            if is_tok:
                total_kept += 1

    out_path = OUTPUT_CONLLU_DIR / fpath.name
    out_path.write_text('\n'.join(out_lines) + '\n', encoding='utf-8')

print(f'\n✅ Done.')
print(f'   Lemmas replaced : {total_replaced}')
print(f'   Kept (X/special): {total_kept}')
print(f'   Updated files written to: {OUTPUT_CONLLU_DIR}')

In [ ]:
# Spot-check: compare original vs. updated lemmas for the first file
first_orig    = sorted(FINAL_CONLLU_DIR.glob('*.conllu'))[0]
first_updated = OUTPUT_CONLLU_DIR / first_orig.name

orig_lines    = first_orig.read_text(encoding='utf-8').splitlines()
updated_lines = first_updated.read_text(encoding='utf-8').splitlines()

diffs = 0
print(f'Spot-check: {first_orig.name}')
print(f'{"FORM":<20} {"OLD LEMMA":<20} {"NEW LEMMA":<20} CHANGED?')
print('-' * 75)
for ol, ul in zip(orig_lines, updated_lines):
    is_tok, fields_o = parse_token_line(ol)
    if not is_tok or '-' in fields_o[0]:
        continue
    fields_u = ul.split('\t')
    old_lem  = fields_o[2]
    new_lem  = fields_u[2]
    changed  = '✓' if old_lem != new_lem else ''
    if diffs < 30:
        print(f'{fields_o[1]:<20} {old_lem:<20} {new_lem:<20} {changed}')
    if changed:
        diffs += 1
print(f'\nTotal changed in this file: {diffs}')